In [1]:
def read_dataframe(filename):
    columns = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance"
    ]

    df = pd.read_parquet(filename, columns=columns)
    df=df.head(1000)  # For testing purposes, limit to first 1000 rows
    df["duration"] = (
        df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    ).dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    numerical = ["trip_distance"]

    df[categorical] = df[categorical].astype(str)

    return df

In [2]:
import numpy as np
import pickle

In [3]:
import pandas as pd
import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Lasso

In [4]:
df_train=read_dataframe("./yellow_tripdata_2026-01.parquet")

In [5]:
categorical=["PULocationID","DOLocationID"]
numerical=["trip_distance"]
dv=DictVectorizer()
train_dict=df_train[categorical+numerical].to_dict(orient="records")
X_train=dv.fit_transform(train_dict)


In [6]:
y_train=df_train["duration"]

In [7]:
X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

In [16]:
with mlflow.start_run():
    mlflow.set_tag("developper","hakim")
    alpha=.002
    mlflow.log_param("alpha",alpha)
    lr=Lasso(alpha=alpha)
    lr.fit(X_train,y_train)
    y_pred=lr.predict(X_train)
    mae=mean_absolute_error(y_train,y_pred)
    mlflow.log_metric("mae",mae)

In [10]:
with open("../models/lin_reg.bin","wb") as f_out:
    pickle.dump((dv,lr),f_out)

In [12]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("new")

<Experiment: artifact_location='/workspaces/MLOPs/01-intro/mlruns/1', creation_time=1789148677294, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789148677294, lifecycle_stage='active', name='new', tags={}, trace_location=None, workspace='default'>